# FuelEcon_R — Reusable Template

Drop-in pipeline for **any** product-year file that looks like EPA `vehicles.csv`: one row per model-year configuration, a time column, an efficiency column, a size column, a maker column, and a fuel / powertrain flag.

Rename columns in the CONFIG cell and run.


In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(readr)
theme_set(theme_minimal())

# ---- CONFIG ----
path          <- "data/vehicles.csv"
time_col      <- "year"
y_col         <- "comb08"      # efficiency
size_col      <- "displ"       # engine / capacity
maker_col     <- "make"
fuel_col      <- "fuelType1"
keep_fuels    <- c("Regular Gasoline", "Premium Gasoline", "Midgrade Gasoline")
exclude_tech  <- c("Hybrid", "Plug-in Hybrid", "EV")  # atvType values to drop
tech_col      <- "atvType"
trans_col     <- "trany"
book_window   <- c(1984, 2014) # years for "present every year" intersect


In [ ]:
raw <- read_csv(path, show_col_types = FALSE)
stopifnot(all(c(time_col, y_col, maker_col) %in% names(raw)))

df <- raw %>%
  rename(
    .time  = !!time_col,
    .y     = !!y_col,
    .maker = !!maker_col
  )
if (size_col %in% names(raw)) df$.size <- as.numeric(raw[[size_col]])
if (fuel_col %in% names(raw)) df$.fuel <- raw[[fuel_col]]
if (tech_col %in% names(raw)) df$.tech <- raw[[tech_col]]
if (trans_col %in% names(raw)) {
  df$.trans <- ifelse(substr(raw[[trans_col]], 1, 4) == "Auto", "Auto", "Manual")
}

cat("rows", nrow(df), "time", min(df$.time), "-", max(df$.time), "\n")


In [ ]:
# Year means — all vs filtered
all_yr <- df %>% group_by(.time) %>% summarise(avg = mean(.y, na.rm = TRUE), series = "All")

core <- df
if (".fuel" %in% names(df)) core <- filter(core, .fuel %in% keep_fuels)
if (".tech" %in% names(df)) core <- filter(core, is.na(.tech) | !(.tech %in% exclude_tech))

core_yr <- core %>% group_by(.time) %>% summarise(avg = mean(.y, na.rm = TRUE), series = "Core")

bind_rows(all_yr, core_yr) %>%
  ggplot(aes(.time, avg, color = series)) +
  geom_line() + geom_point(size = 1.2) +
  labs(x = "Year", y = "Mean efficiency", title = "All series vs filtered core")


In [ ]:
# Size vs efficiency + dual facet if .size exists
if (".size" %in% names(df)) {
  print(
    ggplot(core, aes(.size, .y)) +
      geom_point(alpha = 0.15) + geom_smooth() +
      labs(title = "Size vs efficiency (core)")
  )
  dual <- core %>%
    group_by(.time) %>%
    summarise(efficiency = mean(.y, na.rm = TRUE),
              size = mean(.size, na.rm = TRUE)) %>%
    pivot_longer(-.time)
  print(
    ggplot(dual, aes(.time, value)) +
      geom_point() + geom_smooth() +
      facet_wrap(~name, ncol = 1, scales = "free_y")
  )
}


In [ ]:
# Makes present in every year of book_window
w <- core %>% filter(.time >= book_window[1], .time <= book_window[2])
uniq <- split(w$.maker, w$.time)
common <- Reduce(intersect, lapply(uniq, unique))
print(sort(common))

core %>%
  filter(.maker %in% common) %>%
  group_by(.time, .maker) %>%
  summarise(avg = mean(.y, na.rm = TRUE)) %>%
  ggplot(aes(.time, avg)) +
  geom_line() +
  facet_wrap(~.maker) +
  labs(title = "Common makers across the window")


## How to adapt

1. Point `path` at the new CSV.  
2. Map four columns.  
3. Edit `keep_fuels` / `exclude_tech` so “core” means what *your* audience calls the baseline product.  
4. Re-run the simulation knobs from the Solution notebook on this `core` frame.  
5. Rewrite the four audience paragraphs; do not copy FuelEcon wording into a banking or housing file.
